# Chapter 12 Lab — Agent Communication

**Book:** *Agentic AI: Build, Ship, Portfolio*  
**Chapter 12:** Agent Communication

---

When multiple agents collaborate, the hardest problem is not making each agent
smart — it is making them *understand each other*. This lab explores
communication patterns, message formats, protocols, and conflict resolution
strategies for multi-agent systems.

| Section | What You Build |
|---------|---------------|
| 1 | Setup and imports |
| 2 | Why Communication Is Hard — miscommunication demo |
| 3 | Message Passing Patterns — direct, broadcast, pub/sub |
| 4 | Shared State — a thread-safe state store for agents |
| 5 | Structured Message Formats — Pydantic message schemas |
| 6 | Agent Protocols — request/response and negotiation |
| 7 | Conversation Management — multi-agent conversation orchestration |
| 8 | Conflict Resolution — handling disagreements |
| 9 | Exercises |

> **Prerequisites:** Python 3.10+. No API keys needed — this lab focuses on
> communication infrastructure, not LLM calls.

## 1 — Setup

In [ ]:
%pip install pydantic --quiet

In [ ]:
from __future__ import annotations

import json
import threading
import time
import uuid
from abc import ABC, abstractmethod
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Any, Callable, Optional

from pydantic import BaseModel, Field

print("All imports loaded.")

---
## 2 — Why Communication Is Hard

Before building infrastructure, let's see what goes wrong when agents
communicate with *unstructured* free-text. Two agents try to coordinate on a
task, but their messages are ambiguous — and things fall apart quickly.

In [ ]:
@dataclass
class NaiveAgent:
    """An agent that communicates via unstructured strings."""
    name: str
    inbox: list[str] = field(default_factory=list)

    def send(self, recipient: "NaiveAgent", message: str) -> None:
        print(f"  [{self.name} -> {recipient.name}] {message}")
        recipient.inbox.append(message)

    def process_inbox(self) -> list[str]:
        """Interpret messages — but with no schema, parsing is fragile."""
        results = []
        for msg in self.inbox:
            if "done" in msg.lower():
                results.append(f"{self.name} thinks task is COMPLETE")
            elif "help" in msg.lower():
                results.append(f"{self.name} thinks someone needs HELP")
            elif "ready" in msg.lower():
                results.append(f"{self.name} thinks sender is READY")
            else:
                results.append(f"{self.name} CANNOT UNDERSTAND: '{msg}'")
        self.inbox.clear()
        return results


# ── Demo: ambiguity causes miscommunication ──
alice = NaiveAgent("Alice")
bob = NaiveAgent("Bob")

print("=== Round 1: Alice tells Bob she finished analysis ===")
alice.send(bob, "Analysis is done, but the report still needs work.")
for r in bob.process_inbox():
    print(f"  >> {r}")
# Bob sees "done" and thinks the ENTIRE task is complete!

print("\n=== Round 2: Bob asks for status ===")
bob.send(alice, "Can you provide an update on the deliverables?")
for r in alice.process_inbox():
    print(f"  >> {r}")
# Alice has no keyword match — message is lost.

print("\n=== Takeaway ===")
print("Without structured messages, agents misinterpret intent, lose")
print("information, and cannot reliably coordinate.")

---
## 3 — Message Passing Patterns

We will implement three fundamental patterns:

| Pattern | Description |
|---------|------------|
| **Direct** | Point-to-point: one sender, one receiver |
| **Broadcast** | One sender, all receivers |
| **Pub/Sub** | Agents subscribe to topics; messages are routed by topic |

In [ ]:
@dataclass
class Message:
    """A lightweight message envelope."""
    sender: str
    recipient: str          # "*" for broadcast
    topic: str = "general"
    body: Any = None
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    msg_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])

    def __str__(self) -> str:
        return f"[{self.msg_id}] {self.sender}->{self.recipient} ({self.topic}): {self.body}"


@dataclass
class CommunicatingAgent:
    """Agent with an inbox that can receive Messages."""
    name: str
    inbox: list[Message] = field(default_factory=list)

    def receive(self, message: Message) -> None:
        self.inbox.append(message)

    def read_inbox(self) -> list[Message]:
        msgs = list(self.inbox)
        self.inbox.clear()
        return msgs


print("Message and CommunicatingAgent defined.")

In [ ]:
class MessageRouter:
    """Routes messages using direct, broadcast, and pub/sub patterns."""

    def __init__(self):
        self._agents: dict[str, CommunicatingAgent] = {}
        self._subscriptions: dict[str, list[str]] = defaultdict(list)  # topic -> [agent names]
        self._log: list[Message] = []

    def register(self, agent: CommunicatingAgent) -> None:
        self._agents[agent.name] = agent
        print(f"  Registered agent: {agent.name}")

    def subscribe(self, agent_name: str, topic: str) -> None:
        if agent_name not in self._subscriptions[topic]:
            self._subscriptions[topic].append(agent_name)
            print(f"  {agent_name} subscribed to '{topic}'")

    # ── Direct send ──
    def send_direct(self, message: Message) -> bool:
        self._log.append(message)
        if message.recipient in self._agents:
            self._agents[message.recipient].receive(message)
            return True
        print(f"  WARNING: recipient '{message.recipient}' not found")
        return False

    # ── Broadcast ──
    def broadcast(self, message: Message) -> int:
        message.recipient = "*"
        self._log.append(message)
        count = 0
        for name, agent in self._agents.items():
            if name != message.sender:  # don't send to self
                agent.receive(message)
                count += 1
        return count

    # ── Pub/Sub ──
    def publish(self, message: Message) -> int:
        self._log.append(message)
        subscribers = self._subscriptions.get(message.topic, [])
        count = 0
        for name in subscribers:
            if name != message.sender and name in self._agents:
                self._agents[name].receive(message)
                count += 1
        return count

    def get_log(self) -> list[Message]:
        return list(self._log)


print("MessageRouter ready.")

In [ ]:
# ── Demo: all three patterns ──

router = MessageRouter()

planner = CommunicatingAgent("Planner")
researcher = CommunicatingAgent("Researcher")
writer = CommunicatingAgent("Writer")
reviewer = CommunicatingAgent("Reviewer")

for a in [planner, researcher, writer, reviewer]:
    router.register(a)

# 1) Direct
print("\n--- Direct Message ---")
msg = Message(sender="Planner", recipient="Researcher", topic="task",
              body={"action": "research", "query": "latest RAG techniques"})
router.send_direct(msg)
for m in researcher.read_inbox():
    print(f"  Researcher received: {m}")

# 2) Broadcast
print("\n--- Broadcast ---")
msg = Message(sender="Planner", recipient="*", topic="announcement",
              body="Project kickoff! Everyone stand by.")
n = router.broadcast(msg)
print(f"  Delivered to {n} agents")
for a in [researcher, writer, reviewer]:
    for m in a.read_inbox():
        print(f"  {a.name} got: {m.body}")

# 3) Pub/Sub
print("\n--- Pub/Sub ---")
router.subscribe("Writer", "research-results")
router.subscribe("Reviewer", "research-results")

msg = Message(sender="Researcher", recipient="", topic="research-results",
              body={"findings": ["Hybrid search outperforms keyword-only by 23%"]})
n = router.publish(msg)
print(f"  Published to {n} subscribers")
for a in [writer, reviewer]:
    for m in a.read_inbox():
        print(f"  {a.name} received: {m.body}")

---
## 4 — Shared State

Sometimes message passing is overkill. Agents need to read and write to a
shared blackboard — a **shared state store**. This is the pattern used by
frameworks like LangGraph (state graph) and CrewAI (shared context).

Our implementation is thread-safe and tracks a full change history.

In [ ]:
@dataclass
class StateChange:
    """Record of a single state mutation."""
    agent: str
    key: str
    old_value: Any
    new_value: Any
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class SharedState:
    """Thread-safe key-value store with change tracking."""

    def __init__(self):
        self._state: dict[str, Any] = {}
        self._history: list[StateChange] = []
        self._lock = threading.Lock()
        self._watchers: dict[str, list[Callable]] = defaultdict(list)

    def get(self, key: str, default: Any = None) -> Any:
        with self._lock:
            return self._state.get(key, default)

    def set(self, key: str, value: Any, agent: str = "system") -> None:
        with self._lock:
            old = self._state.get(key)
            self._state[key] = value
            change = StateChange(agent=agent, key=key, old_value=old, new_value=value)
            self._history.append(change)
        # Notify watchers outside lock
        for cb in self._watchers.get(key, []):
            cb(change)

    def watch(self, key: str, callback: Callable[[StateChange], None]) -> None:
        """Register a callback for changes to a specific key."""
        self._watchers[key].append(callback)

    def snapshot(self) -> dict[str, Any]:
        with self._lock:
            return dict(self._state)

    def get_history(self, key: str | None = None) -> list[StateChange]:
        with self._lock:
            if key:
                return [h for h in self._history if h.key == key]
            return list(self._history)


print("SharedState defined.")

In [ ]:
# ── Demo: agents reading/writing shared state ──

state = SharedState()

# Watcher callback
def on_status_change(change: StateChange):
    print(f"  [WATCHER] '{change.key}' changed by {change.agent}: "
          f"{change.old_value!r} -> {change.new_value!r}")

state.watch("project_status", on_status_change)

# Planner sets initial state
state.set("project_status", "planning", agent="Planner")
state.set("research_notes", [], agent="Planner")

# Researcher appends findings
notes = state.get("research_notes", [])
notes.append("Found 5 relevant papers on agent communication.")
state.set("research_notes", notes, agent="Researcher")
state.set("project_status", "research_complete", agent="Researcher")

# Writer reads state
print(f"\nWriter sees status  : {state.get('project_status')}")
print(f"Writer sees notes   : {state.get('research_notes')}")

# Full history
print("\n--- Change History ---")
for h in state.get_history():
    print(f"  {h.agent:>12} set '{h.key}' = {h.new_value!r}")

---
## 5 — Structured Message Formats

The miscommunication demo in Section 2 showed why free-text messages fail.
The fix is **structured message schemas**. We use Pydantic models so that
every message is validated at creation time.

In [ ]:
class MessageType(str, Enum):
    """All recognized message types in our multi-agent system."""
    REQUEST = "request"
    RESPONSE = "response"
    INFORM = "inform"
    PROPOSE = "propose"
    ACCEPT = "accept"
    REJECT = "reject"
    QUERY = "query"
    ERROR = "error"


class Priority(str, Enum):
    LOW = "low"
    NORMAL = "normal"
    HIGH = "high"
    CRITICAL = "critical"


class StructuredMessage(BaseModel):
    """A validated, self-describing message for agent communication."""
    msg_id: str = Field(default_factory=lambda: uuid.uuid4().hex[:8])
    msg_type: MessageType
    sender: str
    recipient: str
    topic: str = "general"
    priority: Priority = Priority.NORMAL
    body: dict[str, Any] = Field(default_factory=dict)
    in_reply_to: Optional[str] = None     # msg_id of the message this replies to
    timestamp: str = Field(default_factory=lambda: datetime.now().isoformat())

    def summary(self) -> str:
        return (f"[{self.msg_id}] {self.msg_type.value.upper()} "
                f"{self.sender}->{self.recipient} topic={self.topic} "
                f"priority={self.priority.value}")


print("StructuredMessage schema:")
print(json.dumps(StructuredMessage.model_json_schema(), indent=2)[:600], "...")

In [ ]:
# ── Demo: create and validate structured messages ──

# Valid message
req = StructuredMessage(
    msg_type=MessageType.REQUEST,
    sender="Planner",
    recipient="Researcher",
    topic="research",
    priority=Priority.HIGH,
    body={"action": "find_papers", "query": "agent communication patterns", "max_results": 10},
)
print("Valid request:")
print(f"  {req.summary()}")
print(f"  Body: {req.body}")

# Reply linking back to the request
resp = StructuredMessage(
    msg_type=MessageType.RESPONSE,
    sender="Researcher",
    recipient="Planner",
    topic="research",
    body={"papers_found": 7, "top_result": "Multi-Agent Communication Survey 2025"},
    in_reply_to=req.msg_id,
)
print(f"\nResponse (replies to {resp.in_reply_to}):")
print(f"  {resp.summary()}")

# Invalid message — validation catches the error
print("\nAttempting invalid message type...")
try:
    bad = StructuredMessage(
        msg_type="SHOUT",  # not in MessageType enum
        sender="Alice",
        recipient="Bob",
    )
except Exception as e:
    print(f"  Validation error: {e}")

---
## 6 — Agent Protocols

A **protocol** defines the *sequence* of messages exchanged between agents.
We implement two protocols:

1. **Request/Response** — agent A asks, agent B answers.
2. **Negotiation** (Contract Net) — agent A issues a call for proposals,
   agents bid, agent A selects the best bid.

In [ ]:
class ProtocolAgent:
    """Agent that can participate in structured protocols."""

    def __init__(self, name: str):
        self.name = name
        self.inbox: list[StructuredMessage] = []
        self.sent: list[StructuredMessage] = []
        self._handlers: dict[MessageType, Callable] = {}

    def on(self, msg_type: MessageType, handler: Callable) -> None:
        """Register a handler for a message type."""
        self._handlers[msg_type] = handler

    def receive(self, message: StructuredMessage) -> Optional[StructuredMessage]:
        """Receive a message and dispatch to the registered handler."""
        self.inbox.append(message)
        handler = self._handlers.get(message.msg_type)
        if handler:
            reply = handler(self, message)
            if reply:
                self.sent.append(reply)
            return reply
        return None

    def send(self, recipient: "ProtocolAgent", message: StructuredMessage) -> Optional[StructuredMessage]:
        """Send a message and return the recipient's reply (if any)."""
        self.sent.append(message)
        return recipient.receive(message)


print("ProtocolAgent defined.")

### 6a — Request / Response Protocol

In [ ]:
def handle_request(agent: ProtocolAgent, msg: StructuredMessage) -> StructuredMessage:
    """Simple handler: echo the request body back with a 'processed' flag."""
    print(f"  {agent.name} processing request: {msg.body}")
    return StructuredMessage(
        msg_type=MessageType.RESPONSE,
        sender=agent.name,
        recipient=msg.sender,
        topic=msg.topic,
        body={"status": "completed", "original_action": msg.body.get("action"), "result": "42"},
        in_reply_to=msg.msg_id,
    )


# ── Demo ──
client_agent = ProtocolAgent("Client")
server_agent = ProtocolAgent("Server")
server_agent.on(MessageType.REQUEST, handle_request)

request = StructuredMessage(
    msg_type=MessageType.REQUEST,
    sender="Client",
    recipient="Server",
    topic="computation",
    body={"action": "compute", "expression": "6 * 7"},
)

print("=== Request/Response Protocol ===")
print(f"Sending: {request.summary()}")
reply = client_agent.send(server_agent, request)
print(f"Reply  : {reply.summary()}")
print(f"Body   : {reply.body}")

### 6b — Negotiation Protocol (Contract Net)

In [ ]:
@dataclass
class Proposal:
    """A bid from a contractor agent."""
    agent_name: str
    cost: float
    estimated_time: float  # seconds
    confidence: float      # 0-1


class ContractNetProtocol:
    """
    Contract Net: a manager broadcasts a call for proposals (CFP),
    contractors bid, the manager selects the best bid.
    """

    def __init__(self, manager_name: str):
        self.manager_name = manager_name
        self.proposals: list[Proposal] = []
        self.log: list[str] = []

    def call_for_proposals(self, task: dict) -> StructuredMessage:
        """Create a CFP message."""
        cfp = StructuredMessage(
            msg_type=MessageType.REQUEST,
            sender=self.manager_name,
            recipient="*",
            topic="cfp",
            priority=Priority.HIGH,
            body={"task": task},
        )
        self.log.append(f"CFP issued: {task}")
        return cfp

    def submit_proposal(self, proposal: Proposal) -> None:
        self.proposals.append(proposal)
        self.log.append(f"Proposal from {proposal.agent_name}: "
                        f"cost={proposal.cost}, time={proposal.estimated_time}s, "
                        f"confidence={proposal.confidence}")

    def evaluate_proposals(self) -> tuple[Optional[Proposal], list[StructuredMessage]]:
        """Select the best proposal using a weighted score."""
        if not self.proposals:
            return None, []

        # Score: lower cost better, lower time better, higher confidence better
        max_cost = max(p.cost for p in self.proposals)
        max_time = max(p.estimated_time for p in self.proposals)

        def score(p: Proposal) -> float:
            cost_score = 1 - (p.cost / max_cost) if max_cost > 0 else 1
            time_score = 1 - (p.estimated_time / max_time) if max_time > 0 else 1
            return 0.3 * cost_score + 0.3 * time_score + 0.4 * p.confidence

        scored = [(p, score(p)) for p in self.proposals]
        scored.sort(key=lambda x: x[1], reverse=True)

        winner = scored[0][0]
        self.log.append(f"Winner: {winner.agent_name} (score={scored[0][1]:.2f})")

        # Build accept/reject messages
        messages = []
        for p, s in scored:
            msg_type = MessageType.ACCEPT if p is winner else MessageType.REJECT
            messages.append(StructuredMessage(
                msg_type=msg_type,
                sender=self.manager_name,
                recipient=p.agent_name,
                topic="cfp-result",
                body={"score": round(s, 3), "selected": p is winner},
            ))
        return winner, messages


print("ContractNetProtocol defined.")

In [ ]:
# ── Demo: Contract Net Negotiation ──

protocol = ContractNetProtocol(manager_name="Manager")

# Step 1: Manager issues CFP
cfp = protocol.call_for_proposals({"task": "write_summary", "pages": 5})
print(f"CFP: {cfp.summary()}\n")

# Step 2: Contractors submit bids
protocol.submit_proposal(Proposal("WriterA", cost=50, estimated_time=3600, confidence=0.9))
protocol.submit_proposal(Proposal("WriterB", cost=30, estimated_time=7200, confidence=0.7))
protocol.submit_proposal(Proposal("WriterC", cost=80, estimated_time=1800, confidence=0.95))

# Step 3: Evaluate
winner, messages = protocol.evaluate_proposals()
print(f"Selected contractor: {winner.agent_name}\n")

print("--- Decision Messages ---")
for m in messages:
    status = "ACCEPTED" if m.msg_type == MessageType.ACCEPT else "rejected"
    print(f"  {m.recipient:>10}: {status} (score={m.body['score']})")

print("\n--- Protocol Log ---")
for entry in protocol.log:
    print(f"  {entry}")

---
## 7 — Conversation Management

In multi-agent systems, we need to track multiple parallel **conversations**
(threads). A `ConversationManager` keeps messages organized by conversation ID
and enforces turn order.

In [ ]:
@dataclass
class Conversation:
    """A single conversation thread between agents."""
    conv_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    participants: list[str] = field(default_factory=list)
    messages: list[StructuredMessage] = field(default_factory=list)
    status: str = "active"  # active | paused | closed
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())

    def add_message(self, message: StructuredMessage) -> None:
        self.messages.append(message)

    def summary(self) -> str:
        return (f"Conv[{self.conv_id}] status={self.status} "
                f"participants={self.participants} msgs={len(self.messages)}")


class ConversationManager:
    """Manages multiple concurrent conversations."""

    def __init__(self):
        self._conversations: dict[str, Conversation] = {}

    def create(self, participants: list[str]) -> Conversation:
        conv = Conversation(participants=participants)
        self._conversations[conv.conv_id] = conv
        return conv

    def get(self, conv_id: str) -> Optional[Conversation]:
        return self._conversations.get(conv_id)

    def add_message(self, conv_id: str, message: StructuredMessage) -> bool:
        conv = self.get(conv_id)
        if not conv or conv.status != "active":
            return False
        if message.sender not in conv.participants:
            return False
        conv.add_message(message)
        return True

    def close(self, conv_id: str) -> None:
        conv = self.get(conv_id)
        if conv:
            conv.status = "closed"

    def get_transcript(self, conv_id: str) -> list[str]:
        conv = self.get(conv_id)
        if not conv:
            return []
        return [f"[{m.msg_type.value}] {m.sender}: {m.body}" for m in conv.messages]

    def list_conversations(self) -> list[str]:
        return [c.summary() for c in self._conversations.values()]


print("Conversation and ConversationManager defined.")

In [ ]:
# ── Demo: multi-agent conversation ──

cm = ConversationManager()

# Conversation 1: Planner <-> Researcher
conv1 = cm.create(["Planner", "Researcher"])
print(f"Created: {conv1.summary()}")

cm.add_message(conv1.conv_id, StructuredMessage(
    msg_type=MessageType.REQUEST, sender="Planner", recipient="Researcher",
    topic="research", body={"query": "agent communication frameworks"}
))
cm.add_message(conv1.conv_id, StructuredMessage(
    msg_type=MessageType.RESPONSE, sender="Researcher", recipient="Planner",
    topic="research", body={"frameworks": ["FIPA ACL", "KQML", "AgentSpeak"]}
))
cm.add_message(conv1.conv_id, StructuredMessage(
    msg_type=MessageType.INFORM, sender="Planner", recipient="Researcher",
    topic="research", body={"feedback": "Great, also look into modern Python solutions"}
))

# Conversation 2: Writer <-> Reviewer
conv2 = cm.create(["Writer", "Reviewer"])
cm.add_message(conv2.conv_id, StructuredMessage(
    msg_type=MessageType.PROPOSE, sender="Writer", recipient="Reviewer",
    topic="review", body={"draft_section": "Introduction", "word_count": 500}
))
cm.add_message(conv2.conv_id, StructuredMessage(
    msg_type=MessageType.RESPONSE, sender="Reviewer", recipient="Writer",
    topic="review", body={"verdict": "needs_revision", "comments": ["Too verbose", "Add examples"]}
))

# Unauthorized participant — should be rejected
result = cm.add_message(conv1.conv_id, StructuredMessage(
    msg_type=MessageType.INFORM, sender="Writer", recipient="Planner",
    topic="research", body={"note": "I want to join!"}
))
print(f"\nWriter tried to join conv1: accepted={result}")

# Print transcripts
for conv_id, label in [(conv1.conv_id, "Planner-Researcher"), (conv2.conv_id, "Writer-Reviewer")]:
    print(f"\n--- Transcript: {label} ---")
    for line in cm.get_transcript(conv_id):
        print(f"  {line}")

# Close conv1
cm.close(conv1.conv_id)
print(f"\nAll conversations:")
for s in cm.list_conversations():
    print(f"  {s}")

---
## 8 — Conflict Resolution

When agents disagree — e.g., two agents propose contradictory edits — we need
a resolution strategy. We implement three:

| Strategy | How It Works |
|----------|-------------|
| **Priority** | The agent with the higher role wins |
| **Voting** | Majority vote among agents |
| **Merge** | Combine non-conflicting parts, flag remaining conflicts |

In [ ]:
class ConflictType(str, Enum):
    VALUE = "value"           # agents propose different values for same key
    ORDERING = "ordering"     # agents disagree on task order
    RESOURCE = "resource"     # agents compete for the same resource


@dataclass
class Conflict:
    """A detected disagreement between agents."""
    conflict_type: ConflictType
    key: str
    positions: dict[str, Any]  # agent_name -> proposed value
    context: str = ""

    def __str__(self) -> str:
        pos = ", ".join(f"{a}={v!r}" for a, v in self.positions.items())
        return f"Conflict({self.conflict_type.value}, key='{self.key}', {pos})"


@dataclass
class Resolution:
    """The result of resolving a conflict."""
    strategy: str
    resolved_value: Any
    explanation: str
    conflict: Conflict


print("Conflict and Resolution defined.")

In [ ]:
class ConflictResolver:
    """Resolves conflicts between agents using pluggable strategies."""

    # Agent priority hierarchy (higher number = more authority)
    DEFAULT_PRIORITIES = {
        "Manager": 100,
        "Reviewer": 80,
        "Planner": 70,
        "Researcher": 50,
        "Writer": 50,
    }

    def __init__(self, priorities: dict[str, int] | None = None):
        self.priorities = priorities or self.DEFAULT_PRIORITIES

    # ── Strategy 1: Priority-based ──
    def resolve_by_priority(self, conflict: Conflict) -> Resolution:
        ranked = sorted(
            conflict.positions.items(),
            key=lambda x: self.priorities.get(x[0], 0),
            reverse=True,
        )
        winner_name, winner_value = ranked[0]
        return Resolution(
            strategy="priority",
            resolved_value=winner_value,
            explanation=f"{winner_name} has highest priority ({self.priorities.get(winner_name, 0)})",
            conflict=conflict,
        )

    # ── Strategy 2: Voting ──
    def resolve_by_voting(self, conflict: Conflict, votes: dict[str, Any] | None = None) -> Resolution:
        """Each agent casts a vote. If votes not provided, agents vote for their own position."""
        ballot = votes or conflict.positions
        # Count which value has the most votes
        tally: dict[str, int] = defaultdict(int)
        for value in ballot.values():
            tally[str(value)] += 1

        winner_value = max(tally, key=lambda v: tally[v])
        return Resolution(
            strategy="voting",
            resolved_value=winner_value,
            explanation=f"Value '{winner_value}' won with {tally[winner_value]} votes (tally: {dict(tally)})",
            conflict=conflict,
        )

    # ── Strategy 3: Merge ──
    def resolve_by_merge(self, conflict: Conflict) -> Resolution:
        """
        If positions are dicts, merge non-conflicting keys.
        For conflicting keys, keep all values in a list.
        """
        positions = conflict.positions
        if not all(isinstance(v, dict) for v in positions.values()):
            # Fallback: collect all values
            return Resolution(
                strategy="merge",
                resolved_value=list(positions.values()),
                explanation="Non-dict values — collected all positions into a list",
                conflict=conflict,
            )

        merged = {}
        conflicts_remaining = []
        all_keys = set()
        for d in positions.values():
            all_keys.update(d.keys())

        for key in all_keys:
            values = {agent: d[key] for agent, d in positions.items() if key in d}
            unique = set(str(v) for v in values.values())
            if len(unique) == 1:
                merged[key] = list(values.values())[0]  # all agree
            else:
                merged[key] = {"CONFLICT": values}
                conflicts_remaining.append(key)

        explanation = (
            f"Merged {len(all_keys)} keys. "
            f"{len(all_keys) - len(conflicts_remaining)} agreed, "
            f"{len(conflicts_remaining)} still conflicting: {conflicts_remaining}"
        )
        return Resolution(
            strategy="merge",
            resolved_value=merged,
            explanation=explanation,
            conflict=conflict,
        )


print("ConflictResolver defined.")

In [ ]:
# ── Demo: resolve conflicts ──

resolver = ConflictResolver()

# Conflict: two agents disagree on the title
c1 = Conflict(
    conflict_type=ConflictType.VALUE,
    key="report_title",
    positions={
        "Writer":   "Agent Communication: A Practical Guide",
        "Reviewer": "Multi-Agent Communication Patterns in Practice",
    },
    context="Both agents proposed different report titles."
)
print(f"Conflict: {c1}\n")

# Strategy 1: Priority
r1 = resolver.resolve_by_priority(c1)
print(f"Priority resolution : {r1.resolved_value!r}")
print(f"  Explanation: {r1.explanation}\n")

# Strategy 2: Voting (with a third agent casting the deciding vote)
r2 = resolver.resolve_by_voting(c1, votes={
    "Writer": "Agent Communication: A Practical Guide",
    "Reviewer": "Multi-Agent Communication Patterns in Practice",
    "Planner": "Agent Communication: A Practical Guide",  # sides with Writer
})
print(f"Voting resolution   : {r2.resolved_value!r}")
print(f"  Explanation: {r2.explanation}\n")

# Conflict: agents propose different section contents (dict merge)
c2 = Conflict(
    conflict_type=ConflictType.VALUE,
    key="section_plan",
    positions={
        "Writer": {
            "intro": "Start with a story",
            "body": "Three main points",
            "conclusion": "Call to action",
        },
        "Reviewer": {
            "intro": "Start with a story",        # agrees
            "body": "Five main points with data",  # disagrees
            "conclusion": "Summary and next steps", # disagrees
            "appendix": "References",               # new key
        },
    },
)
r3 = resolver.resolve_by_merge(c2)
print(f"Merge resolution:")
print(f"  {json.dumps(r3.resolved_value, indent=4)}")
print(f"  Explanation: {r3.explanation}")

### Putting It All Together — A Mini Multi-Agent Workflow

Let's run a small end-to-end scenario that combines message passing, shared
state, structured messages, and conflict resolution.

In [ ]:
print("=" * 60)
print("MINI WORKFLOW: Collaborative Report Planning")
print("=" * 60)

# Infrastructure
workflow_state = SharedState()
workflow_cm = ConversationManager()
workflow_resolver = ConflictResolver()

# Step 1: Planner sets the task
print("\n[Step 1] Planner sets initial task...")
workflow_state.set("task", {"type": "report", "topic": "Agent Communication"}, agent="Planner")
workflow_state.set("phase", "research", agent="Planner")

# Step 2: Researcher and Writer discuss in a conversation
print("\n[Step 2] Researcher and Writer discuss outline...")
outline_conv = workflow_cm.create(["Researcher", "Writer"])

workflow_cm.add_message(outline_conv.conv_id, StructuredMessage(
    msg_type=MessageType.PROPOSE, sender="Researcher", recipient="Writer",
    topic="outline",
    body={"sections": ["Intro", "Patterns", "Protocols", "Conclusion"]}
))
workflow_cm.add_message(outline_conv.conv_id, StructuredMessage(
    msg_type=MessageType.PROPOSE, sender="Writer", recipient="Researcher",
    topic="outline",
    body={"sections": ["Intro", "Patterns", "Protocols", "Case Study", "Conclusion"]}
))

# Step 3: Conflict detected — different outlines
print("\n[Step 3] Conflict detected on outline...")
outline_conflict = Conflict(
    conflict_type=ConflictType.ORDERING,
    key="outline",
    positions={
        "Researcher": ["Intro", "Patterns", "Protocols", "Conclusion"],
        "Writer":     ["Intro", "Patterns", "Protocols", "Case Study", "Conclusion"],
    },
)
resolution = workflow_resolver.resolve_by_priority(outline_conflict)
# Writer and Researcher have equal priority, so whichever is first alphabetically
# Let's use voting with Planner as tiebreaker
resolution = workflow_resolver.resolve_by_voting(outline_conflict, votes={
    "Researcher": ["Intro", "Patterns", "Protocols", "Conclusion"],
    "Writer":     ["Intro", "Patterns", "Protocols", "Case Study", "Conclusion"],
    "Planner":    ["Intro", "Patterns", "Protocols", "Case Study", "Conclusion"],
})
print(f"  Resolution: {resolution.explanation}")

# Step 4: Store resolved outline in shared state
print("\n[Step 4] Storing resolved outline...")
# Parse the winning value back (it was stringified by voting)
final_outline = ["Intro", "Patterns", "Protocols", "Case Study", "Conclusion"]
workflow_state.set("outline", final_outline, agent="ConflictResolver")
workflow_state.set("phase", "writing", agent="Planner")

# Final state
print("\n[Final State]")
for k, v in workflow_state.snapshot().items():
    print(f"  {k}: {v}")

print("\n[Conversation Transcript]")
for line in workflow_cm.get_transcript(outline_conv.conv_id):
    print(f"  {line}")

---
## 9 — Exercises

### Exercise 1 — Conceptual (no code)

Consider a multi-agent system where four agents collaborate on code review:
- **Linter** checks style
- **SecurityScanner** checks vulnerabilities
- **Tester** runs unit tests
- **Reviewer** gives the final verdict

**Questions:**
1. Which communication pattern (direct, broadcast, pub/sub) would you use
   between each pair of agents and why?
2. What happens if Linter and SecurityScanner produce contradictory
   recommendations (e.g., Linter says "remove this code" and SecurityScanner
   says "this code is a critical security check")? Which conflict resolution
   strategy would you apply?
3. Should these agents use message passing, shared state, or both? Justify
   your answer.

### Exercise 2 — Coding: Build a Message Bus

Implement a `MessageBus` class that combines the router and conversation
manager into a single unified interface. Requirements:

- `send(message)` automatically routes based on recipient (direct if named,
  broadcast if `"*"`, pub/sub if recipient is empty but topic is set)
- Every message is logged to an internal conversation by topic
- `get_topic_history(topic)` returns all messages for a topic
- `get_agent_history(agent_name)` returns all messages sent or received by an agent

Starter code is provided below.

In [ ]:
class MessageBus:
    """Unified message bus combining routing and conversation tracking."""

    def __init__(self):
        self._agents: dict[str, CommunicatingAgent] = {}
        self._subscriptions: dict[str, list[str]] = defaultdict(list)
        self._all_messages: list[StructuredMessage] = []
        self._topic_index: dict[str, list[StructuredMessage]] = defaultdict(list)
        self._agent_index: dict[str, list[StructuredMessage]] = defaultdict(list)

    def register(self, agent: CommunicatingAgent) -> None:
        self._agents[agent.name] = agent

    def subscribe(self, agent_name: str, topic: str) -> None:
        if agent_name not in self._subscriptions[topic]:
            self._subscriptions[topic].append(agent_name)

    def send(self, message: StructuredMessage) -> int:
        """
        Route a message and return the number of agents it was delivered to.

        TODO: Implement routing logic:
        - If recipient is "*": broadcast to all agents except sender
        - If recipient is "" (empty) and topic is set: publish to topic subscribers
        - Otherwise: direct send to named recipient

        Don't forget to index every message for get_topic_history / get_agent_history.
        """
        # YOUR CODE HERE
        pass

    def get_topic_history(self, topic: str) -> list[StructuredMessage]:
        """Return all messages for a given topic."""
        # YOUR CODE HERE
        pass

    def get_agent_history(self, agent_name: str) -> list[StructuredMessage]:
        """Return all messages sent or received by an agent."""
        # YOUR CODE HERE
        pass


# ── Test your implementation ──
# Uncomment the lines below once you have filled in the methods.

# bus = MessageBus()
# a1 = CommunicatingAgent("Alpha")
# a2 = CommunicatingAgent("Beta")
# a3 = CommunicatingAgent("Gamma")
# for a in [a1, a2, a3]:
#     bus.register(a)
# bus.subscribe("Beta", "alerts")
# bus.subscribe("Gamma", "alerts")
#
# # Direct
# bus.send(StructuredMessage(msg_type=MessageType.INFORM, sender="Alpha", recipient="Beta",
#                            topic="task", body={"info": "direct hello"}))
# # Broadcast
# bus.send(StructuredMessage(msg_type=MessageType.INFORM, sender="Alpha", recipient="*",
#                            topic="general", body={"info": "broadcast hello"}))
# # Pub/Sub
# bus.send(StructuredMessage(msg_type=MessageType.INFORM, sender="Alpha", recipient="",
#                            topic="alerts", body={"info": "alert!"}))
#
# print(f"Beta inbox: {len(a2.inbox)} messages")
# print(f"Gamma inbox: {len(a3.inbox)} messages")
# print(f"Topic 'alerts' history: {len(bus.get_topic_history('alerts'))} messages")
# print(f"Alpha's history: {len(bus.get_agent_history('Alpha'))} messages")

### Exercise 3 — Design: Protocol for Collaborative Writing Agents

Design (on paper or in markdown) a communication protocol for a team of
agents that collaboratively write a technical document. The team includes:

- **Outliner** — creates the document structure
- **Drafter** — writes initial content for each section
- **Editor** — improves clarity, grammar, and flow
- **FactChecker** — verifies claims and adds citations
- **Coordinator** — manages workflow and resolves disputes

Your protocol design should specify:

1. **Message types** needed beyond the ones defined in this lab
2. **Conversation flows** — draw or describe the sequence of messages for:
   - Starting a new document
   - Completing and handing off a section
   - Requesting a revision
   - Resolving a factual disagreement
3. **Shared state keys** — what should live in the shared state vs. be
   communicated via messages?
4. **Error handling** — what happens if an agent fails or times out?

Write your design in the cell below as structured markdown.

*Your design here...*

---

**End of Chapter 12 Lab.** You now have working implementations of the core
communication primitives for multi-agent systems: message passing (direct,
broadcast, pub/sub), shared state, structured messages, protocols
(request/response, Contract Net), conversation management, and conflict
resolution. These patterns form the foundation for building reliable
multi-agent applications.